# 07 — Fabrication and physical balance

This notebook reads the current CAD export. If it is absent, run the printed
command and reopen the notebook. It never substitutes analytic geometry for a
missing part build. Inspect preview images, calibration agreement, printer fit
and both balance alternatives before selecting a mechanical revision.


## CAD evidence versus analytic geometry

**Inputs:** current `outputs/parts/manifest.json` and exported meshes/STEP files.
**Implementation:** `uavdesign/fab/build.py`, airframe/cage generators and shared
`fab/geometry.py`. Manufacturing settings are in `fabrication.yaml`.

The analytic wing defines planform, airfoil blend and twist. The CAD build adds
shell walls, finite trailing edges, print splits, joiner paths, clearances,
cutouts and purchased-component envelopes. Assembly STEP is in the aircraft
frame ($x$ aft, $y$ right, $z$ up); each print STL has its own bed-oriented frame.
Never infer an assembled location from the STL's print coordinates.

The manifest mass entries estimate deposited material and hardware masses.
They are **not measured finished-part weights or slicer-consumption estimates**.
The balance calculation uses $x_{CG}=\sum m_i x_i/\sum m_i$ with the declared
component locations. Battery positions must satisfy both the allowed envelope
and installed-component interference checks. A target marked unreachable must
remain an open design issue even when all parts individually fit the printer.

Plain-text equation: `CG_x = sum(component_mass * component_x) / sum(component_mass)`.

**External review:** compare the preview, assembly and source dimensions; inspect
bonding access, connector routing, screw access, retention, cooling and actual
vendor envelope tolerances. Notebook 10 hashes the current files and reports
manufacturing evidence; a hash confirms identity, not strength or printability.


In [ ]:
import os
for key in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[key] = "1"
os.environ.setdefault("MPLCONFIGDIR", "/tmp/uavdesign-notebook-matplotlib")
"""Locate the repo root so `import uavdesign` works without `pip install -e .`."""
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "uavdesign").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

PLATFORM_DIR = REPO_ROOT / "platforms" / "survey_wing"
FIG_DIR = REPO_ROOT / "outputs" / "figures" / "survey_wing"
FIG_DIR.mkdir(parents=True, exist_ok=True)

import yaml
import numpy as np
import matplotlib.pyplot as plt

import json
import pandas as pd
from IPython.display import display, Markdown, Image
from uavdesign import design as D
from uavdesign import survey as sv
from uavdesign.atmosphere import G
studies = yaml.safe_load((PLATFORM_DIR / "notebook_studies.yaml").read_text())
d = D.run(solve_stability=True)
s = d.design["survey"]
cam = sv.Camera.from_dict(s["camera"])
from dataclasses import replace
acquisition_cam = replace(cam, max_fps=min(cam.max_fps, s["acquisition_rate_cap_hz"]))
pl = d.design["planform"]
req = d.design["requirements"]

def save(fig, name):
    fig.tight_layout()
    path = FIG_DIR / name
    fig.savefig(path, dpi=140)
    display(Image(filename=str(path)))
    plt.close(fig)


In [ ]:
manifest_path=REPO_ROOT/"outputs/parts/manifest.json"
manifest=json.loads(manifest_path.read_text()) if manifest_path.exists() else None
if manifest is None:
    print("No parts manifest. Run: python scripts/build_parts.py")
else:
    display(pd.DataFrame(manifest["parts"]))
    display(pd.DataFrame([manifest["balance"]]))
    print("Manifest sizing summary:",manifest.get("design",manifest.get("sizing",{})))


In [ ]:
if manifest is not None:
    parts=pd.DataFrame(manifest["parts"])
    if "mass_g" in parts and "group" in parts:
        display(parts.groupby("group")["mass_g"].sum().rename("cad_printed_mass_g").to_frame())
    display(pd.Series(d.selected.masses_g,name="sizing_mass_g").to_frame())
    for path in sorted(manifest_path.parent.glob("preview*.png")):
        display(Markdown(f"### {path.name}"));display(Image(filename=str(path)))
    balance=manifest["balance"]
    fig,ax=plt.subplots(figsize=(9,3))
    for name,key in (("CG","cg_x_mm"),("Neutral point","x_np_mm"),("Battery","battery_x_mm")):
        if key in balance: ax.axvline(balance[key],label=f"{name}: {balance[key]:.1f} mm")
    ax.axvline(d.x_cg_target_mm,color="black",ls="--",label="Sizing target CG")
    ax.set(xlabel="x aft of root leading edge [mm]",yticks=[]);ax.legend()
    save(fig,"07_balance.png")
    if "contingency_g" in balance:
        display(pd.DataFrame(D.balance_alternatives(d,balance)))
    else:
        print("Legacy balance lacks contingency_g: rebuild parts before comparing alternatives.")
